In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta import DeltaTable

In [0]:
df = spark.read.table("databricks_cat.silver.customers")
df.display()

In [0]:
df.dropDuplicates(subset=["customer_id"])

In [0]:
##Check Table exists:

table_exists = spark.catalog.tableExists("databricks_cat.gold.customers")
table_exists    


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

if table_exists:
    df_old = spark.read.table("databricks_cat.gold.customers")

    # Get max surrogate key safely
    max_surrogate_key = df_old.select(F.max("DimCustomerKey").alias("max_dim")).first()["max_dim"]

else:
    max_surrogate_key = 0

    # Define schema directly instead of SELECT 0 WHERE 1=0
    schema = StructType([
        StructField("DimCustomerKey", IntegerType(), True),
        StructField("customer_id", StringType(), True),
        StructField("created_date", TimestampType(), True),
        StructField("updated_date", TimestampType(), True),
    ])
    df_old = spark.createDataFrame([], schema)

# Common join logic (no duplication)
df_join = (
    df.join(df_old, df.customer_id == df_old.customer_id, "left")
      .select(
          df["*"],
          df_old.DimCustomerKey,
          df_old.created_date,
          df_old.updated_date
      )
)

display(df_join.limit(2))

In [0]:
print(max_surrogate_key)

In [0]:
df_new = df_join.filter(col("DimCustomerKey").isNull())
display(df_new.limit(2))



In [0]:
df_old = df_join.filter(col("DimCustomerKey").isNotNull())
df_old = df_old.withColumn("updated_date", current_timestamp())
display(df_old)

In [0]:
## Add Surrogate Key to the df_new
df_new = df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1)+ lit(max_surrogate_key))
df_new = df_new.withColumn("created_date", current_timestamp())
df_new = df_new.withColumn("updated_date", current_timestamp())

In [0]:
display(df_new.orderBy("customer_id",desc=True).limit(2))

## **Inserting** new records

In [0]:
df_all = df_old.unionByName(df_new)
display(df_all.limit(10))

# UPSERT THE records

In [0]:

if table_exists:
    table_target = DeltaTable.forName(spark,"databricks_cat.gold.customers")
    {
    table_target.alias("tr").merge(df_all.alias("sr"), "tr.customer_id = sr.customer_id") \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll()\
    .execute()
    }

else:
    df_all.write.mode("overwrite").format("delta").save("abfss://gold@databricksgvse2e.dfs.core.windows.net/customers")
    spark.sql("""
              CREATE TABLE IF NOT EXISTS databricks_cat.gold.customers
               USING DELTA
               LOCATION 'abfss://gold@databricksgvse2e.dfs.core.windows.net/customers'
               """)


In [0]:
df = spark.read.table("databricks_cat.gold.customers")
display(df)


In [0]:
print(df.count())